# Deliverable 3 - Final Model, Hyperparameter Tuning, and Kaggle Submission Bonus

## Members:
- Mustafa Al-Youzbaki  
- Gabriel Burrows 
- Queenie Li 
- Michelle Shi 

# Full Pipeline Construction

### Initialize Data and Import Libraries
**Source**: [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview)

First we import necessary libraries and load the dataset.

Based on our **Deliverable 1 findings** and further refinements, we will:
1. Drop outliers identified in EDA and those with extreme `GrLivArea` (> 4500 sqft).
2. Apply a log-transformation to the target variable `SalePrice`.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Scikit-learn Pipeline and Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, KFold, GridSearchCV

# Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, make_scorer

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# Load Data
df_train = pd.read_csv('train.csv')

# --- 1. Outlier Removal ---
# Removing houses with extreme GrLivArea (> 4500) as seen in outliers analysis
df_train = df_train[df_train.GrLivArea < 4500]
df_train.reset_index(drop=True, inplace=True)

# Removing specific outliers identified in Deliverable 1
outlier_ids = [30, 88, 462, 631, 692, 1183, 1299, 1322]
df_train = df_train[~df_train['Id'].isin(outlier_ids)]

# --- 2. Target Transformation ---
# Log-transform SalePrice to normalize distribution (np.log1p applies log(1+x))
df_train['SalePrice'] = np.log1p(df_train['SalePrice'])

print(f"Data loaded. Shape: {df_train.shape}")

### Feature Engineering

To improve on regression results over the previous two deliverables, we decided to implement new a feature engineering function which implements strategies such as:
- **Grouped Imputation:** Filling `LotFrontage` based on Neighborhood median.
- **New Features:** Creating aggregate metrics like `TotalSF` and `Total_Bathrooms`.
- **Type Conversion:** Converting numerical categories (e.g., `MSSubClass`) to string.

These new aggregate features were decided upon reflection on our EDA findings from deliverable 1.

In [ ]:
def feature_engineering(df):
    """
    Custom function to handle specific imputation and feature creation strategies.
    Applied to both training and testing data before the sklearn pipeline.
    """
    df = df.copy()
    
    # --- A. Data Type Conversions ---
    # Some numerical features are actually categorical
    for col in ['MSSubClass', 'YrSold', 'MoSold']:
        df[col] = df[col].astype(str)
        
    # --- B. Specific Imputations ---
    # Grouped Imputation: LotFrontage by Neighborhood median
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
    
    # MSZoning by MSSubClass mode
    df['MSZoning'] = df.groupby('MSSubClass')['MSZoning'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "RL"))
    
    # Fill specific categoricals with Mode or Specific Values
    df['Functional'] = df['Functional'].fillna('Typ')
    df['Electrical'] = df['Electrical'].fillna('SBrkr')
    df['KitchenQual'] = df['KitchenQual'].fillna('TA')
    df['Exterior1st'] = df['Exterior1st'].fillna(df['Exterior1st'].mode()[0])
    df['Exterior2nd'] = df['Exterior2nd'].fillna(df['Exterior2nd'].mode()[0])
    df['SaleType'] = df['SaleType'].fillna(df['SaleType'].mode()[0])
    
    # Fill 'None' or 0 for missing amenities (Pool, Garage, Basement)
    df['PoolQC'] = df['PoolQC'].fillna('None')
    for col in ['GarageYrBlt', 'GarageArea', 'GarageCars']:
        df[col] = df[col].fillna(0)
    for col in ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']:
        df[col] = df[col].fillna('None')
    for col in ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']:
        df[col] = df[col].fillna('None')

    # --- C. Feature Creation ---
    # Total Square Footage
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    
    # Total Bathrooms (counting half baths as 0.5)
    df['Total_Bathrooms'] = (df['FullBath'] + (0.5 * df['HalfBath']) + 
                             df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath']))
    
    # Total Porch Area
    df['Total_porch_sf'] = (df['OpenPorchSF'] + df['3SsnPorch'] +
                              df['EnclosedPorch'] + df['ScreenPorch'] +
                              df['WoodDeckSF'])
    
    # Simplified Boolean Features
    df['haspool'] = df['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
    df['has2ndfloor'] = df['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
    df['hasgarage'] = df['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
    df['hasbsmt'] = df['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
    df['hasfireplace'] = df['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)
    
    # Drop Utilities and Street as they have almost zero variance
    df = df.drop(['Utilities', 'Street', 'PoolQC'], axis=1, errors='ignore')
    
    return df

# Apply Feature Engineering
df_train_processed = feature_engineering(df_train)

# Separate Target and Features
y = df_train_processed['SalePrice']
X = df_train_processed.drop(['SalePrice', 'Id'], axis=1)

print(f"Data Shape after Feature Engineering: {X.shape}")

### Preprocessing Pipeline
We construct a scikit-learn `ColumnTransformer` to handle the remaining preprocessing.

* **Numerical Features:** 
    * `SimpleImputer`: Handles any remaining missing values (median).
    * `StandardScaler`: Normalizes the data (Mean=0, Std=1), which is crucial for linear models like Ridge.
* **Categorical Features:** 
    * `SimpleImputer`: Fills remaining missing values with "None".
    * `OneHotEncoder`: Converts categories to numeric vectors.

In [ ]:
# Identify numerical and categorical columns automatically
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Define Preprocessing Steps
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),      # Handle remaining missing values
    ('scaler', StandardScaler())                        # Scale features
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')), # Handle remaining missing values
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Apply dummy encoding to categorical variables
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline defined.")

### Model Comparision
We compare distinct model families to find the best candidate for tuning:
1.  **Linear Family (Ridge Regression):** A regularized linear model. We use Ridge (L2 regularization) as identified in Deliverable 2's next steps to handle multicollinearity among the many features.
2.  **Ensemble Family (Gradient Boosting):** A tree-based boosting method that often performs well on Kaggle competitions by capturing non-linear interactions automatically.
3.  **Instance-based Family (KNN):** The refined model from Deliverable 2, included here to see how it performs with the new feature engineering and log-transformation.

In [ ]:
# Define Pipelines for three model families

# Model 1: Ridge Regression (Linear Family)
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(random_state=42))
])

# Model 2: Gradient Boosting (Ensemble Family)
gbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(n_estimators=100, random_state=42))
])

# Model 3: KNN (Refined model from Deliverable 2)
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', KNeighborsRegressor(n_neighbors=5))
])

print("Pipelines constructed.")

### Evaluation Metrics (Initial Comparison)
We use 10-fold Cross-Validation to evaluate both models. 
*Note: Since we log-transformed the target, the RMSE calculated here is effectively the **RMSLE** (Root Mean Squared Logarithmic Error), which is the official metric for the Kaggle competition.*

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Define scoring metrics for cross_validate
scoring = {
    'RMSE': 'neg_root_mean_squared_error',  # negative due to sklearn convention
    'MAE': 'neg_mean_absolute_error',
    'R2': 'r2'
}

models = [('Ridge Regression', ridge_pipeline), 
          ('Gradient Boosting', gbm_pipeline),
          ('KNN Regression', knn_pipeline)]

summary_rows = []

print("Performing Cross-Validation...\n")

for name, model in models:
    # Run cross-validation
    cv_results = cross_validate(model, X, y, cv=kf, scoring=scoring, return_train_score=True)
    
    # Create summary row in the exact format from Deliverable 2
    row = {
        'Model': name,
        'Train RMSE Mean': -np.mean(cv_results['train_RMSE']),
        'Train RMSE Std': np.std(cv_results['train_RMSE']),
        'Valid RMSE Mean': -np.mean(cv_results['test_RMSE']),
        'Valid RMSE Std': np.std(cv_results['test_RMSE']),
        'Valid MAE Mean': -np.mean(cv_results['test_MAE']),
        'Valid R2 Mean': np.mean(cv_results['test_R2']),
        'Valid R2 Std': np.std(cv_results['test_R2'])
    }
    summary_rows.append(row)
    print(f"{name} Completed.")

# Convert to DataFrame and display
cv_summary_df = pd.DataFrame(summary_rows).round(4)
display(cv_summary_df)

# Systematic Hyperparameter Tuning of Ridge Regression Model

### Justification
Based on the initial comparison, we will tune the **Ridge Regression** model. 

While Gradient Boosting is powerful, the **Ridge model** provides high interpretability and performed comparably (or better) in our initial baseline tests. Tuning the regularization parameter (`alpha`) is crucial to balance bias and variance, specifically to **prevent overfitting** on this high-dimensional dataset.

### Search Spaces and Setup


In [ ]:
# Define Search Space
# We search alpha values on a logarithmic scale from very weak (0.01) to very strong (100) regularization.
param_grid = {
    'regressor__alpha': np.logspace(-2, 2, 20) # 20 values between 0.01 and 100
}

# Setup GridSearchCV
grid_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, # Use all processors
    return_train_score=True
)

### Execution and Best Hyperparameters

In [ ]:
# Fit the Grid Search
grid_search.fit(X, y)

# Get results
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = -grid_search.best_score_  # Convert back to positive RMSE

# Calculate standard deviation of the best model
best_index = grid_search.best_index_
best_std = grid_search.cv_results_['std_test_score'][best_index]

print(f"Best Hyperparameters: {best_params}")
print(f"Best CV RMSE: {best_score:.4f} (± {best_std:.4f})")

### Mean ± std of CV performance

Looking at the results above, the best Ridge Regression model achieved a mean RMSE of **0.1147 (± 0.0062)**. 

The small variability suggests that the model **generalizes consistently** and is not highly sensitive to how the data is partitioned.

### Comparison with Deliverable 2 baseline

In our Deliverable 2 baseline, we had a mean validation RMSE of around **39,000** and a standard deviation of around **8-10k** relative to 400k prices. While this seems like a huge change compared to our Ridge model's RMSE of 0.1147 and a std of 0.0062, we must note that these two models are on **completely different scales**. Additionally, the preprocessing approach was different between deliverables, as we had found a better way to implement it this time around. Because RMSE depends on the scale of the model's target variable, **any change** in how `y` is defined, filtered, or transformed will change the magnitude of RMSE (as evidently shown here).

However, we can still compare model behaviour:
- The Ridge model has a **much lower variance**, implying higher stability across folds.
- Regularization **reduces overfitting**, compared to the baseline linear regression model.

### Plot/Table Summary (e.g., Error bar plots and validation curves) 

The following code displays the **mean ± std RMSE** for each α tested in the grid.

In [ ]:
# Extract alphas, mean CV scores, and std scores
alphas = grid_search.cv_results_['param_regressor__alpha'].data.astype(float)
mean_scores = -grid_search.cv_results_['mean_test_score']   # convert back to positive RMSE
std_scores = grid_search.cv_results_['std_test_score']

plt.figure(figsize=(8,5))
plt.errorbar(alphas, mean_scores, yerr=std_scores, fmt='o-', capsize=4)
plt.xscale('log')
plt.xlabel('Alpha (Regularization Strength)')
plt.ylabel('CV RMSE')
plt.title('Ridge Regression: Error Bar Plot (Mean ± Std)')
plt.grid(True)
plt.show()

Looking at the graph above, we see that the best alpha value that provides the lowest mean cross-validated RMSE occurs around **α ≈ 10–20**. After that, we see a slight increase in the RMSE as alpha continues to increase past that optimal value, which then indicates mild underfitting.

Next, we show **train vs. validation RMSE** across α:

In [ ]:
from sklearn.model_selection import validation_curve

train_scores, val_scores = validation_curve(
    ridge_pipeline,
    X, y,
    param_name='regressor__alpha',
    param_range=alphas,  # reuse your grid
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

train_mean = -np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = -np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(8,5))
plt.plot(alphas, train_mean, label='Train RMSE')
plt.plot(alphas, val_mean, label='Validation RMSE')
plt.fill_between(alphas, train_mean-train_std, train_mean+train_std, alpha=0.2)
plt.fill_between(alphas, val_mean-val_std, val_mean+val_std, alpha=0.2)
plt.xscale('log')
plt.xlabel('Alpha')
plt.ylabel('RMSE')
plt.title('Validation Curve for Ridge Regression')
plt.legend()
plt.grid(True)
plt.show()

Here, we see a similar story:
- Small alpha values show signs of **overfitting**. This makes sense, as the closer to 0 alpha is, the less regularization the model has - potentially leading to very large coefficients, lower training RMSE but poor generalization, and overall characteristics of an **unregularized linear regression** model.

- Large alpha values show **underfitting**. When alpha becomes too large, all coefficients aggressively shrink toward 0, resulting in an **overly simplified model**.

- The sweet spot for alpha is still around **α ≈ 10–20**, where the validation RMSE is at its lowest. This correlates to what we see in our error bar plot!

Based on these results, we can conclude that **α ≈ 10–20** is the optimal range for our model, as it balances bias and variance while minimizing validation RMSE.

# Final Model Selection and Evaluation

### Final cross validated performance

### Diagnostic visualization (e.g., ROC, confusion matrix, residuals, PDP)

### Interpretation of potential error patterns and limitations

# Integration of TA feedback

### Relevant feedback and how each point was addressed

### Evidence of improvements where appropriate

# Final Reflection

Write a few sentences summarizing:
- what your team learned,
- what was challenging or rewarding,
- one additional improvement you would try if you had more time, and why.


# Kaggle Submission

### Kaggle CSV

In [ ]:
# Load Test Data
df_test = pd.read_csv('test.csv')

# Store Ids for submission
test_ids = df_test['Id']

# Apply the same feature engineering to the test set
df_test_processed = feature_engineering(df_test)

# Make predictions
log_predictions = best_model.predict(df_test_processed)

# Inverse Log Transformation to get real prices
actual_predictions = np.expm1(log_predictions)

# Create Submission DataFrame
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': actual_predictions
})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")

### Kaggle Score (screenshot submitted seperately)

**Final Score:** 0.13392 \
**Leaderboard Rank** 2364/6024 (Top 39%)

### Cross Validation vs Kaggle Performance

# Individual Contributions
## Who Did What


| Group Member | Role |
| --- | --- |
| Mustafa Al-Youzbaki  | |
| Gabriel Burrows | Full Pipeline Construction, Hyperparameter Tuning with Grid Search, General notebook formatting, Kaggle Submission |
| Queenie Li | |
| Michelle Shi  | |

## Checklist

- [X] Two or more model families tested
- [X] Full pipeline implemented
- [X] Hyperparameter tuning with GridSearchCV/RandomizedSearchCV
- [ ] Hyperparameter tuning visualization/summary included 
- [ ] Final model selected + justified
- [ ] Diagnostic visualizations included
- [ ] Feedback from previous deliverables addressed
- [ ] Final reflection completed
- [X] Contributions table included
- [X] (Optional) Kaggle CSV created
- [X] (Optional) Screenshot shows the score and Group Number in Kaggle’s submission
description